In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("LOWKEY - Member 2")
print("Data Preparation Started!")

In [ ]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_file = next(iter(uploaded))

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall('/content/LOWKEY_DATASET')

print("Dataset extracted successfully!")

os.listdir('/content/LOWKEY_DATASET')

In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/LOWKEY_DATASET"

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
events = pd.read_csv(
    "/content/LOWKEY_DATASET/tap_scroll_events.csv"
)

acc = pd.read_csv(
    "/content/LOWKEY_DATASET/accelerometer_data.csv"
)

print("TAP + SCROLL:", events.shape)
print("ACCELEROMETER:", acc.shape)

display(events.head())
display(acc.head())

In [ ]:
print("TAP + SCROLL Missing Values:")
print(events.isnull().sum())

print("\nAccelerometer Missing Values:")
print(acc.isnull().sum())

In [ ]:
print("Event Type Counts:")
print(events['event_type'].value_counts())

print("\nTAP Coordinate Availability:")
print(events[events['event_type'] == 'TAP']['x_position'].notna().sum())

print("\nTotal Accelerometer Records:", len(acc))

In [ ]:
print("TAP + SCROLL Duplicates:")
print(events.duplicated(subset=events.columns.drop('id')).sum())

print("\nAccelerometer Exact Duplicates:")
print(acc.duplicated().sum())

print("\nDuplicate Timestamps:")
print(acc['timestamp'].duplicated().sum())

In [ ]:
events['timestamp'] = events['timestamp'].astype('int64')
acc['timestamp'] = acc['timestamp'].astype('int64')

acc_sorted = acc.sort_values('timestamp')

time_diff = acc_sorted['timestamp'].diff()

print("Zero time differences:", (time_diff == 0).sum())
print("Negative differences:", (time_diff < 0).sum())
print("Gaps above 1 second:", (time_diff > 1000).sum())
print("Gaps above 60 seconds:", (time_diff > 60000).sum())
print("Largest gap (ms):", time_diff.max())

In [ ]:
duplicate_rows = acc[acc.duplicated(keep=False)]

print("Exact duplicate rows:", len(duplicate_rows))
display(duplicate_rows.head(10))

In [ ]:
acc['is_exact_duplicate'] = acc.duplicated(keep='first')

print("Rows flagged as duplicates:", acc['is_exact_duplicate'].sum())
print("Total rows preserved:", len(acc))

In [ ]:
print("TAP/SCROLL range:")
print(pd.to_datetime(events['timestamp'], unit='ms', utc=True).min())
print(pd.to_datetime(events['timestamp'], unit='ms', utc=True).max())

print("\nAccelerometer range:")
print(pd.to_datetime(acc['timestamp'], unit='ms', utc=True).min())
print(pd.to_datetime(acc['timestamp'], unit='ms', utc=True).max())

In [ ]:
for df in [events, acc]:
    df['timestamp_utc'] = pd.to_datetime(
        df['timestamp'], unit='ms', utc=True
    )
    df['timestamp_ist'] = df['timestamp_utc'].dt.tz_convert(
        'Asia/Kolkata'
    )

print("Timestamp conversion completed!")
display(events[['timestamp', 'timestamp_ist']].head())
display(acc[['timestamp', 'timestamp_ist']].head())

In [ ]:
print("Accuracy values:")
print(acc['accuracy'].value_counts())

print("\nAccelerometer magnitude:")
acc['magnitude'] = np.sqrt(
    acc['x']**2 + acc['y']**2 + acc['z']**2
)

display(acc[['x', 'y', 'z', 'magnitude']].describe())

In [ ]:
print("Extreme accelerometer readings:")

print("Magnitude above 20:",
      (acc['magnitude'] > 20).sum())

print("Magnitude above 50:",
      (acc['magnitude'] > 50).sum())

print("Magnitude above 100:",
      (acc['magnitude'] > 100).sum())

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(acc['timestamp_ist'], acc['magnitude'],
         linewidth=0.5)

plt.title("Accelerometer Magnitude Over Time")
plt.xlabel("Timestamp")
plt.ylabel("Magnitude")
plt.grid(True)
plt.show()

In [ ]:
sample = acc.iloc[::1000]

plt.figure(figsize=(12, 5))

plt.plot(sample['timestamp_ist'], sample['x'], label='X')
plt.plot(sample['timestamp_ist'], sample['y'], label='Y')
plt.plot(sample['timestamp_ist'], sample['z'], label='Z')

plt.title("Accelerometer X, Y, Z Readings")
plt.xlabel("Timestamp")
plt.ylabel("Sensor Values")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import os

os.makedirs("LOWKEY_PROCESSED", exist_ok=True)

events.to_csv(
    "LOWKEY_PROCESSED/tap_scroll_audited.csv",
    index=False
)

acc.to_csv(
    "LOWKEY_PROCESSED/accelerometer_audited.csv",
    index=False
)

print("Audited datasets saved successfully!")

In [ ]:
from google.colab import files

files.download("LOWKEY_PROCESSED/tap_scroll_audited.csv")
files.download("LOWKEY_PROCESSED/accelerometer_audited.csv")


In [ ]:
# Remove only exact duplicate records
acc_clean = acc.drop_duplicates().copy()

# Sort by timestamp
acc_clean = acc_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

print("Original records:", len(acc))
print("Cleaned records:", len(acc_clean))
print("Duplicates remaining:", acc_clean.duplicated().sum())

In [ ]:
acc_clean = acc.drop_duplicates(
    subset=['timestamp', 'x', 'y', 'z', 'accuracy']
).copy()

acc_clean = acc_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

print("Original:", len(acc))
print("Cleaned:", len(acc_clean))
print("Removed:", len(acc) - len(acc_clean))

In [ ]:
events_clean = events.copy()

events_clean = events_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

print("Original records:", len(events))
print("Cleaned records:", len(events_clean))

print("Missing values preserved:")
print(events_clean.isnull().sum())

In [ ]:
import os

os.makedirs("LOWKEY_PROCESSED", exist_ok=True)

events_clean.to_csv(
    "LOWKEY_PROCESSED/cleaned_tap_scroll.csv",
    index=False
)

acc_clean.to_csv(
    "LOWKEY_PROCESSED/cleaned_accelerometer.csv",
    index=False
)

print("Both cleaned datasets saved successfully!")

In [ ]:
from google.colab import files

files.download("LOWKEY_PROCESSED/cleaned_tap_scroll.csv")
files.download("LOWKEY_PROCESSED/cleaned_accelerometer.csv")

In [ ]:
# Separate TAP and SCROLL events

tap = events_clean[
    events_clean['event_type'] == 'TAP'
].copy()

scroll = events_clean[
    events_clean['event_type'] == 'SCROLL'
].copy()

print("TAP records:", len(tap))
print("SCROLL records:", len(scroll))

# TAP features
tap_features = tap.groupby(
    pd.Grouper(key='timestamp_ist', freq='10s')
).agg(
    tap_count=('id', 'count'),
    avg_tap_x=('x_position', 'mean'),
    avg_tap_y=('y_position', 'mean')
).reset_index()

# SCROLL features
scroll_features = scroll.groupby(
    pd.Grouper(key='timestamp_ist', freq='10s')
).agg(
    scroll_count=('id', 'count'),
    avg_scroll_x=('scroll_x', 'mean'),
    avg_scroll_y=('scroll_y', 'mean'),
    avg_scroll_delta_y=('scroll_delta_y', 'mean')
).reset_index()

print("Feature engineering started successfully!")

In [ ]:

# Merge TAP and SCROLL features

features = pd.merge(
    tap_features,
    scroll_features,
    on='timestamp_ist',
    how='outer'
)

# Sort by timestamp
features = features.sort_values('timestamp_ist')

# Replace missing values with 0
features = features.fillna(0)

print("Merged features successfully!")
print("Total feature records:", len(features))

# Display first 5 rows
print(features.head())


In [ ]:

# Save extracted features
features.to_csv('tap_scroll_features.csv', index=False)

print("Dataset saved successfully!")


In [ ]:

# Check dataset information
print("Dataset Shape:", features.shape)

print("\nColumn Names:")
print(features.columns.tolist())

print("\nDataset Information:")
print(features.info())

print("\nStatistical Summary:")
print(features.describe())

print("\nMissing Values:")
print(features.isnull().sum())


In [ ]:

# Check extreme values in TAP and SCROLL features

print("TAP X - Highest values:")
print(features.nlargest(10, 'avg_tap_x')[
    ['timestamp_ist', 'avg_tap_x']
])

print("\nSCROLL X - Highest values:")
print(features.nlargest(10, 'avg_scroll_x')[
    ['timestamp_ist', 'avg_scroll_x']
])

print("\nSCROLL DELTA Y - Lowest values:")
print(features.nsmallest(10, 'avg_scroll_delta_y')[
    ['timestamp_ist', 'avg_scroll_delta_y']
])

print("\nPercentile Analysis:")
print(features[
    ['avg_tap_x', 'avg_tap_y',
     'avg_scroll_x', 'avg_scroll_y',
     'avg_scroll_delta_y']
].quantile([0.50, 0.90, 0.95, 0.99, 1.00]))


In [ ]:

# Inspect original SCROLL events with extreme values

extreme_scroll = events_clean[
    (events_clean['event_type'] == 'SCROLL') &
    (
        (events_clean['scroll_x'].abs() > 10000) |
        (events_clean['scroll_delta_y'].abs() > 1000) |
        (events_clean['scroll_y'].abs() > 5000)
    )
].copy()

print("Number of extreme SCROLL events:",
      len(extreme_scroll))

print("\nOriginal extreme SCROLL records:")

print(extreme_scroll.head(20).to_string(index=False))

print("\nOriginal SCROLL statistics:")
print(
    events_clean[
        events_clean['event_type'] == 'SCROLL'
    ][['scroll_x', 'scroll_y', 'scroll_delta_y']].describe()
)


In [ ]:

# 1. Check non-zero scroll values
scroll_data = events_clean[
    events_clean['event_type'] == 'SCROLL'
].copy()

scroll_columns = [
    'scroll_x',
    'scroll_y',
    'scroll_delta_x',
    'scroll_delta_y',
    'max_scroll_x',
    'max_scroll_y'
]

print("NON-ZERO SCROLL VALUE COUNTS")
print((scroll_data[scroll_columns] != 0).sum())

# 2. Check non-zero value distributions
print("\nNON-ZERO SCROLL STATISTICS")

for col in scroll_columns:
    values = scroll_data.loc[scroll_data[col] != 0, col]
    print(f"\n{col}")
    print(values.describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    ))

# 3. Display largest scroll movements
print("\nLARGEST SCROLL DELTA Y EVENTS")

print(
    scroll_data.reindex(
        scroll_data['scroll_delta_y'].abs()
        .nlargest(10).index
    )[
        ['id', 'timestamp_ist', 'scroll_x',
         'scroll_y', 'scroll_delta_y',
         'max_scroll_x', 'max_scroll_y']
    ].to_string(index=False)
)

# 4. Check scroll records around the largest event
print("\nEVENTS AROUND LARGEST SCROLL MOVEMENT")

largest_index = scroll_data[
    'scroll_delta_y'
].abs().idxmax()

position = scroll_data.index.get_loc(largest_index)

print(
    scroll_data.iloc[
        max(0, position-5):position+6
    ][
        ['id', 'timestamp_ist', 'scroll_x',
         'scroll_y', 'scroll_delta_y']
    ].to_string(index=False)
)


In [ ]:

# Create a separate copy for investigation
scroll_check = scroll_data.copy()

# Check horizontal scroll anomalies
invalid_x = scroll_check[
    (scroll_check['scroll_x'] > 10000) |
    (scroll_check['scroll_delta_x'].abs() > 10000)
]

# Check vertical scroll anomalies
invalid_y = scroll_check[
    (scroll_check['scroll_delta_y'] <= -5000) |
    (
        (scroll_check['max_scroll_y'] > 0) &
        (scroll_check['scroll_y'] >
         scroll_check['max_scroll_y'])
    )
]

print("Horizontal suspicious events:", len(invalid_x))
print("Vertical suspicious events:", len(invalid_y))

print("\nSuspicious horizontal records:")
print(
    invalid_x[
        ['id', 'scroll_x', 'scroll_delta_x',
         'max_scroll_x']
    ].head(15).to_string(index=False)
)

print("\nSuspicious vertical records:")
print(
    invalid_y[
        ['id', 'scroll_y', 'scroll_delta_y',
         'max_scroll_y']
    ].head(15).to_string(index=False)
)

# Check repeated extreme values
print("\nRepeated extreme vertical movements:")
print(
    scroll_check[
        scroll_check['scroll_delta_y'] <= -5000
    ]['scroll_delta_y'].value_counts()
)


In [ ]:

# Inspect horizontal anomaly sequence
print("HORIZONTAL SCROLL SEQUENCE")

print(
    scroll_data[
        scroll_data['id'].between(930, 960)
    ][
        ['id', 'timestamp_ist',
         'scroll_x', 'scroll_y',
         'scroll_delta_x', 'scroll_delta_y',
         'max_scroll_x', 'max_scroll_y']
    ].to_string(index=False)
)

# Inspect vertical anomaly sequence
print("\nVERTICAL SCROLL SEQUENCE")

print(
    scroll_data[
        scroll_data['id'].between(4078, 4095)
    ][
        ['id', 'timestamp_ist',
         'scroll_x', 'scroll_y',
         'scroll_delta_x', 'scroll_delta_y',
         'max_scroll_x', 'max_scroll_y']
    ].to_string(index=False)
)

# Check missing maximum scroll information
print("\nMAX SCROLL CHECK")

print(
    "Non-zero scroll_x with zero max_scroll_x:",
    len(scroll_data[
        (scroll_data['scroll_x'] != 0) &
        (scroll_data['max_scroll_x'] == 0)
    ])
)

print(
    "Non-zero scroll_y with zero max_scroll_y:",
    len(scroll_data[
        (scroll_data['scroll_y'] != 0) &
        (scroll_data['max_scroll_y'] == 0)
    ])
)


In [ ]:

# Create a separate copy
events_flagged = events_clean.copy()

# Identify SCROLL events
is_scroll = events_flagged['event_type'] == 'SCROLL'

# Flag suspicious horizontal measurements
events_flagged['horizontal_anomaly'] = (
    is_scroll &
    (
        (events_flagged['scroll_x'].abs() > 10000) |
        (events_flagged['scroll_delta_x'].abs() > 10000)
    )
)

# Flag extreme vertical movements
events_flagged['vertical_anomaly'] = (
    is_scroll &
    (events_flagged['scroll_delta_y'] <= -5000)
)

# Combined anomaly flag
events_flagged['is_anomaly'] = (
    events_flagged['horizontal_anomaly'] |
    events_flagged['vertical_anomaly']
)

# Display results
print("Total original events:", len(events_clean))

print("Horizontal anomalies:",
      events_flagged['horizontal_anomaly'].sum())

print("Vertical anomalies:",
      events_flagged['vertical_anomaly'].sum())

print("Total flagged events:",
      events_flagged['is_anomaly'].sum())

# Save flagged dataset
events_flagged.to_csv(
    'events_with_anomaly_flags.csv',
    index=False
)

print("\nAnomaly-flagged dataset saved successfully!")


In [ ]:

# STEP 15: Recalculate clean features

# 1. Separate TAP and SCROLL events
tap_clean = events_flagged[
    events_flagged['event_type'] == 'TAP'
].copy()

scroll_clean = events_flagged[
    events_flagged['event_type'] == 'SCROLL'
].copy()

# 2. Exclude suspicious horizontal measurements
scroll_clean.loc[
    scroll_clean['scroll_x'].abs() > 10000,
    'scroll_x'
] = float('nan')

# 3. Exclude suspicious vertical movements
scroll_clean.loc[
    scroll_clean['vertical_anomaly'],
    'scroll_delta_y'
] = float('nan')

# 4. Recalculate TAP features
tap_features_clean = tap_clean.groupby(
    pd.Grouper(key='timestamp_ist', freq='10s')
).agg(
    tap_count=('id', 'count'),
    avg_tap_x=('x_position', 'mean'),
    avg_tap_y=('y_position', 'mean')
).reset_index()

# 5. Recalculate SCROLL features
scroll_features_clean = scroll_clean.groupby(
    pd.Grouper(key='timestamp_ist', freq='10s')
).agg(
    scroll_count=('id', 'count'),
    avg_scroll_x=('scroll_x', 'mean'),
    avg_scroll_y=('scroll_y', 'mean'),
    avg_scroll_delta_y=('scroll_delta_y', 'mean')
).reset_index()

# 6. Merge both datasets
features_clean = pd.merge(
    tap_features_clean,
    scroll_features_clean,
    on='timestamp_ist',
    how='outer'
)

features_clean = features_clean.sort_values(
    'timestamp_ist'
).reset_index(drop=True)

# 7. Replace missing event counts with zero
features_clean[['tap_count', 'scroll_count']] = (
    features_clean[['tap_count', 'scroll_count']].fillna(0)
)

# 8. Save cleaned dataset
features_clean.to_csv(
    'cleaned_tap_scroll_features.csv',
    index=False
)

print("Clean feature engineering completed!")
print("Clean dataset shape:", features_clean.shape)

print("\nMissing values:")
print(features_clean.isnull().sum())

print("\nCleaned dataset preview:")
print(features_clean.head())

print("\nCleaned dataset saved successfully!")


In [ ]:

# STEP 16: Prepare ML-ready dataset

ml_features = features_clean.copy()

# Fill missing numerical feature values
feature_columns = [
    'avg_tap_x',
    'avg_tap_y',
    'avg_scroll_x',
    'avg_scroll_y',
    'avg_scroll_delta_y'
]

ml_features[feature_columns] = (
    ml_features[feature_columns].fillna(0)
)

# Check missing values
print("Missing values after preparation:")
print(ml_features.isnull().sum())

# Save ML-ready dataset
ml_features.to_csv(
    'ml_ready_features.csv',
    index=False
)

print("\nML-ready dataset shape:", ml_features.shape)
print("ML-ready dataset saved successfully!")

print("\nDataset preview:")
print(ml_features.head())


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

# Convert timestamp to datetime
ml_features['timestamp_ist'] = pd.to_datetime(
    ml_features['timestamp_ist']
)

# Set timestamp as index
visual_data = ml_features.set_index('timestamp_ist')

# Aggregate activity per minute
minute_data = visual_data[
    ['tap_count', 'scroll_count']
].resample('1min').sum()

# GRAPH 1: TAP Activity
plt.figure(figsize=(14, 5))
plt.plot(
    minute_data.index,
    minute_data['tap_count']
)
plt.title('TAP Activity Over Time')
plt.xlabel('Time')
plt.ylabel('Number of Taps')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig('tap_activity.png')
plt.show()

# GRAPH 2: SCROLL Activity
plt.figure(figsize=(14, 5))
plt.plot(
    minute_data.index,
    minute_data['scroll_count']
)
plt.title('SCROLL Activity Over Time')
plt.xlabel('Time')
plt.ylabel('Number of Scrolls')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig('scroll_activity.png')
plt.show()

# GRAPH 3: TAP vs SCROLL Comparison
plt.figure(figsize=(14, 5))
plt.plot(
    minute_data.index,
    minute_data['tap_count'],
    label='TAP'
)
plt.plot(
    minute_data.index,
    minute_data['scroll_count'],
    label='SCROLL'
)
plt.title('TAP vs SCROLL Activity')
plt.xlabel('Time')
plt.ylabel('Event Count')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.savefig('tap_scroll_comparison.png')
plt.show()

print("All three visualizations created successfully!")


In [ ]:
# STEP 17: Hourly activity visualization

import matplotlib.pyplot as plt

# Convert event counts to integers
ml_features['tap_count'] = ml_features['tap_count'].astype(int)
ml_features['scroll_count'] = ml_features['scroll_count'].astype(int)

# Calculate hourly activity
hourly_activity = (
    ml_features.set_index('timestamp_ist')
    [['tap_count', 'scroll_count']]
    .resample('1h')
    .sum()
)

# Graph 1: Hourly TAP activity
plt.figure(figsize=(14, 5))
plt.plot(
    hourly_activity.index,
    hourly_activity['tap_count'],
    label='TAP Activity'
)
plt.title('Hourly TAP Activity')
plt.xlabel('Time')
plt.ylabel('Number of TAP Events')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

# Graph 2: Hourly SCROLL activity
plt.figure(figsize=(14, 5))
plt.plot(
    hourly_activity.index,
    hourly_activity['scroll_count'],
    label='SCROLL Activity'
)
plt.title('Hourly SCROLL Activity')
plt.xlabel('Time')
plt.ylabel('Number of SCROLL Events')
plt.legend()
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

print("Hourly activity visualization completed!")
print(hourly_activity.head(10))

In [ ]:
# STEP 18: User Activity Analysis

# Total activity
total_taps = ml_features['tap_count'].sum()
total_scrolls = ml_features['scroll_count'].sum()

# Hourly averages
avg_taps = hourly_activity['tap_count'].mean()
avg_scrolls = hourly_activity['scroll_count'].mean()

# Most active hours
most_active_tap_hour = hourly_activity['tap_count'].idxmax()
most_active_scroll_hour = hourly_activity['scroll_count'].idxmax()

# Display results
print("========== USER ACTIVITY ANALYSIS ==========")

print("\n1. Total TAP Events:", total_taps)
print("2. Total SCROLL Events:", total_scrolls)

print("\n3. Average TAP Events per Hour:", round(avg_taps, 2))
print("4. Average SCROLL Events per Hour:", round(avg_scrolls, 2))

print("\n5. Most Active TAP Hour:", most_active_tap_hour)
print("6. Most Active SCROLL Hour:", most_active_scroll_hour)

print("\n7. Maximum TAP Events in One Hour:",
      hourly_activity['tap_count'].max())

print("8. Maximum SCROLL Events in One Hour:",
      hourly_activity['scroll_count'].max())

print("\nActivity analysis completed!")

In [ ]:
# STEP 19: Daily Activity Analysis

import matplotlib.pyplot as plt

# Calculate daily activity
daily_activity = (
    ml_features.set_index('timestamp_ist')
    [['tap_count', 'scroll_count']]
    .resample('1D')
    .sum()
)

# Print daily activity
print("========== DAILY ACTIVITY ==========")
print(daily_activity)

# Daily TAP activity graph
plt.figure(figsize=(12, 5))
plt.bar(daily_activity.index, daily_activity['tap_count'])
plt.title('Daily TAP Activity')
plt.xlabel('Date')
plt.ylabel('Total TAP Events')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

# Daily SCROLL activity graph
plt.figure(figsize=(12, 5))
plt.bar(daily_activity.index, daily_activity['scroll_count'])
plt.title('Daily SCROLL Activity')
plt.xlabel('Date')
plt.ylabel('Total SCROLL Events')
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.tight_layout()
plt.show()

print("\nDaily activity analysis completed!")

In [ ]:
# STEP 20: TAP and SCROLL Relationship Analysis

import matplotlib.pyplot as plt

# Calculate correlation
correlation = ml_features[
    ['tap_count', 'scroll_count']
].corr()

print("========== CORRELATION ANALYSIS ==========")
print(correlation)

print(
    "\nCorrelation between TAP and SCROLL:",
    round(correlation.loc['tap_count', 'scroll_count'], 3)
)

# Scatter plot
plt.figure(figsize=(8, 5))

plt.scatter(
    ml_features['tap_count'],
    ml_features['scroll_count'],
    alpha=0.4
)

plt.title('Relationship Between TAP and SCROLL Activity')
plt.xlabel('TAP Count per 10 Seconds')
plt.ylabel('SCROLL Count per 10 Seconds')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nCorrelation analysis completed!")

In [ ]:
# STEP 21: Behavioral Feature Engineering

# Create a separate dataset
behavior_features = ml_features.copy()

# Extract time-based features
behavior_features['hour'] = (
    behavior_features['timestamp_ist'].dt.hour
)

behavior_features['day_of_week'] = (
    behavior_features['timestamp_ist'].dt.dayofweek
)

behavior_features['is_weekend'] = (
    behavior_features['day_of_week'].isin([5, 6]).astype(int)
)

# Calculate rolling activity over 1 minute (6 intervals)
behavior_features['tap_1min'] = (
    behavior_features['tap_count']
    .rolling(window=6, min_periods=1).sum()
)

behavior_features['scroll_1min'] = (
    behavior_features['scroll_count']
    .rolling(window=6, min_periods=1).sum()
)

# Calculate rolling activity over 5 minutes (30 intervals)
behavior_features['tap_5min'] = (
    behavior_features['tap_count']
    .rolling(window=30, min_periods=1).sum()
)

behavior_features['scroll_5min'] = (
    behavior_features['scroll_count']
    .rolling(window=30, min_periods=1).sum()
)

# Save the new dataset
behavior_features.to_csv(
    'behavioral_features.csv',
    index=False
)

print("========== BEHAVIORAL FEATURE ENGINEERING ==========")
print("Dataset shape:", behavior_features.shape)

print("\nNew features:")
print([
    'hour',
    'day_of_week',
    'is_weekend',
    'tap_1min',
    'scroll_1min',
    'tap_5min',
    'scroll_5min'
])

print("\nMissing values:")
print(behavior_features.isnull().sum())

print("\nDataset preview:")
print(behavior_features.head())

print("\nBehavioral dataset saved successfully!")

In [ ]:
# STEP 22: Prepare data for anomaly detection

from sklearn.preprocessing import StandardScaler

# Select features for the model
selected_features = [
    'tap_count',
    'scroll_count',
    'avg_tap_x',
    'avg_tap_y',
    'avg_scroll_x',
    'avg_scroll_y',
    'avg_scroll_delta_y',
    'hour',
    'day_of_week',
    'is_weekend',
    'tap_1min',
    'scroll_1min',
    'tap_5min',
    'scroll_5min'
]

# Create feature matrix
X = behavior_features[selected_features].copy()

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("========== MODEL DATA PREPARATION ==========")

print("Original feature shape:", X.shape)
print("Scaled feature shape:", X_scaled.shape)

print("\nMissing values:", X.isnull().sum().sum())

print("\nData standardization completed!")

In [ ]:
# STEP 23: Train Anomaly Detection Model

from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=100,
    contamination=0.02,
    random_state=42
)

model.fit(X_scaled)

# Predict anomalies
predictions = model.predict(X_scaled)

# Convert predictions
behavior_features['anomaly'] = predictions
behavior_features['anomaly'] = (
    behavior_features['anomaly'].map({1: 0, -1: 1})
)

# Count results
print("========== MODEL RESULTS ==========")
print("Normal activities:",
      (behavior_features['anomaly'] == 0).sum())

print("Anomalous activities:",
      (behavior_features['anomaly'] == 1).sum())

print("Model training completed!")

In [ ]:
# STEP 24: Anomaly Visualization

import matplotlib.pyplot as plt

# Count normal and anomalous records
anomaly_counts = behavior_features['anomaly'].value_counts()

# Pie chart
plt.figure(figsize=(7, 7))
plt.pie(
    [anomaly_counts.get(0, 0), anomaly_counts.get(1, 0)],
    labels=['Normal Activity', 'Anomalous Activity'],
    autopct='%1.2f%%',
    startangle=90
)
plt.title('User Activity Anomaly Detection')
plt.show()

# Scatter plot
plt.figure(figsize=(10, 5))

normal = behavior_features[
    behavior_features['anomaly'] == 0
]
anomalies = behavior_features[
    behavior_features['anomaly'] == 1
]

plt.scatter(
    normal['tap_count'],
    normal['scroll_count'],
    label='Normal',
    alpha=0.3
)

plt.scatter(
    anomalies['tap_count'],
    anomalies['scroll_count'],
    label='Anomaly',
    alpha=0.7
)

plt.xlabel('TAP Count')
plt.ylabel('SCROLL Count')
plt.title('Normal vs Anomalous User Activity')
plt.legend()
plt.grid(True)
plt.show()

print("Anomaly visualization completed!")

In [ ]:
# STEP 26: Save Model and Scaler

import joblib

# Save trained Isolation Forest model
joblib.dump(model, 'isolation_forest_model.pkl')

# Save scaler
joblib.dump(scaler, 'feature_scaler.pkl')

# Save feature names
joblib.dump(selected_features, 'selected_features.pkl')

# Save final dataset again
behavior_features.to_csv(
    'final_anomaly_detection_results.csv',
    index=False
)

print("========== PROJECT FILES SAVED ==========")
print("1. isolation_forest_model.pkl")
print("2. feature_scaler.pkl")
print("3. selected_features.pkl")
print("4. final_anomaly_detection_results.csv")
print("\nAll files saved successfully!")

In [ ]:
print("========== PROJECT SUMMARY ==========")
print("Project: User Behavior Anomaly Detection")
print("Algorithm: Isolation Forest")
print("Total records:", len(behavior_features))
print("Features used:", len(selected_features))
print("Normal records:",
      (behavior_features['anomaly'] == 0).sum())
print("Anomalies:",
      (behavior_features['anomaly'] == 1).sum())
print("Anomaly percentage:",
      round(behavior_features['anomaly'].mean() * 100, 2), "%")

In [ ]:
# STEP 28: Analyze detected anomalies

anomalies_df = behavior_features[
    behavior_features['anomaly'] == 1
].copy()

print("========== ANOMALY ANALYSIS ==========")

print("\nAnomaly statistics:")
print(
    anomalies_df[
        ['tap_count', 'scroll_count',
         'tap_1min', 'scroll_1min',
         'tap_5min', 'scroll_5min']
    ].describe()
)

print("\nFirst 15 detected anomalies:")
print(
    anomalies_df[
        ['timestamp_ist', 'tap_count',
         'scroll_count', 'tap_1min',
         'scroll_1min', 'anomaly']
    ].head(15)
)

# Save anomaly records
anomalies_df.to_csv(
    'detected_anomalies.csv',
    index=False
)

print("\nAnomaly records saved successfully!")

In [ ]:
# STEP 29: Normal vs Anomalous Comparison

comparison = behavior_features.groupby('anomaly')[
    ['tap_count', 'scroll_count',
     'tap_1min', 'scroll_1min',
     'tap_5min', 'scroll_5min']
].mean()

comparison.index = ['Normal', 'Anomalous']

print("========== NORMAL VS ANOMALOUS ==========")
print(comparison.round(2))

# Visual comparison
comparison.T.plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title('Average Normal vs Anomalous User Activity')
plt.xlabel('Behavioral Features')
plt.ylabel('Average Activity')
plt.xticks(rotation=45)
plt.legend(title='Activity Type')
plt.tight_layout()
plt.show()

print("\nComparison completed!")

In [ ]:
# STEP 30: Check timestamp intervals

time_gaps = (
    behavior_features['timestamp_ist']
    .sort_values()
    .diff()
    .value_counts()
)

print("========== TIMESTAMP INTERVAL CHECK ==========")
print("Most common time intervals:")
print(time_gaps.head(10))

print("\nTotal timestamps:", len(behavior_features))
print("Unique timestamps:",
      behavior_features['timestamp_ist'].nunique())

print("\nValidation completed!")

In [ ]:
# STEP 31: Export project results

# Daily activity
daily_activity.to_csv('daily_activity_summary.csv')

# Hourly activity
hourly_activity.to_csv('hourly_activity_summary.csv')

# Normal vs anomalous comparison
comparison.to_csv('normal_vs_anomalous.csv')

# Detected anomalies
anomalies_df.to_csv('detected_anomalies.csv', index=False)

# Final complete dataset
behavior_features.to_csv(
    'final_anomaly_detection_results.csv',
    index=False
)

print("ALL PROJECT RESULTS EXPORTED SUCCESSFULLY!")

In [ ]:
# STEP 32: Accelerometer Feature Engineering

import numpy as np
import pandas as pd

# Use original audited accelerometer records.
# Do not remove repeated timestamps just because timestamps match.
acc_features_source = acc.copy()

# Sort records by timestamp
acc_features_source = acc_features_source.sort_values(
    'timestamp_ist'
).copy()

# Calculate magnitude if it does not already exist
if 'magnitude' not in acc_features_source.columns:
    acc_features_source['magnitude'] = np.sqrt(
        acc_features_source['x']**2 +
        acc_features_source['y']**2 +
        acc_features_source['z']**2
    )

# Group accelerometer readings into 10-second windows
acc_features = acc_features_source.groupby(
    pd.Grouper(key='timestamp_ist', freq='10s')
).agg(
    acc_sample_count=('x', 'count'),

    x_mean=('x', 'mean'),
    x_std=('x', 'std'),
    x_min=('x', 'min'),
    x_max=('x', 'max'),

    y_mean=('y', 'mean'),
    y_std=('y', 'std'),
    y_min=('y', 'min'),
    y_max=('y', 'max'),

    z_mean=('z', 'mean'),
    z_std=('z', 'std'),
    z_min=('z', 'min'),
    z_max=('z', 'max'),

    magnitude_mean=('magnitude', 'mean'),
    magnitude_std=('magnitude', 'std'),
    magnitude_min=('magnitude', 'min'),
    magnitude_max=('magnitude', 'max'),

    magnitude_rms=('magnitude',
                   lambda s: np.sqrt(np.mean(s**2))),

    acc_time_span_ms=('timestamp',
                      lambda s: s.max() - s.min())
).reset_index()

# Calculate axis ranges
acc_features['x_range'] = (
    acc_features['x_max'] - acc_features['x_min']
)

acc_features['y_range'] = (
    acc_features['y_max'] - acc_features['y_min']
)

acc_features['z_range'] = (
    acc_features['z_max'] - acc_features['z_min']
)

# Save accelerometer features
acc_features.to_csv(
    'accelerometer_10sec_features.csv',
    index=False
)

print("Accelerometer feature engineering completed!")
print("Dataset shape:", acc_features.shape)

print("\nColumns:")
print(acc_features.columns.tolist())

print("\nMissing values:")
print(acc_features.isnull().sum())

print("\nPreview:")
display(acc_features.head())

print("\nStatistical summary:")
display(acc_features.describe())

In [ ]:
# STEP 33: TAP Coordinate Availability

events_clean['coordinate_available'] = (
    events_clean['event_type'].eq('TAP') &
    events_clean['x_position'].notna() &
    events_clean['y_position'].notna()
).astype(int)

print("Coordinate availability:")
print(events_clean[
    events_clean['event_type'] == 'TAP'
]['coordinate_available'].value_counts())

print("\nMissing TAP coordinates:")
print(
    events_clean[
        events_clean['event_type'] == 'TAP'
    ][['x_position', 'y_position']].isnull().sum()
)

events_clean.to_csv(
    'cleaned_tap_scroll.csv',
    index=False
)

print("\nTAP coordinate flag saved!")

In [ ]:
print("TAP/SCROLL range:")
print(events_clean['timestamp_ist'].min())
print(events_clean['timestamp_ist'].max())

print("\nAccelerometer range:")
print(acc_features['timestamp_ist'].min())
print(acc_features['timestamp_ist'].max())

In [ ]:
# STEP 34: Merge all modalities

start_time = max(
    events_clean['timestamp_ist'].min(),
    acc_features['timestamp_ist'].min()
)

end_time = min(
    events_clean['timestamp_ist'].max(),
    acc_features['timestamp_ist'].max()
)

tap_scroll_features = features_clean.copy()

tap_scroll_features = tap_scroll_features[
    (tap_scroll_features['timestamp_ist'] >= start_time) &
    (tap_scroll_features['timestamp_ist'] <= end_time)
]

acc_overlap = acc_features[
    (acc_features['timestamp_ist'] >= start_time) &
    (acc_features['timestamp_ist'] <= end_time)
]

print("Overlap start:", start_time)
print("Overlap end:", end_time)
print("TAP/SCROLL shape:", tap_scroll_features.shape)
print("Accelerometer shape:", acc_overlap.shape)

In [ ]:
# STEP 35: Merge all features

lowkey_features = pd.merge(
    tap_scroll_features,
    acc_overlap,
    on='timestamp_ist',
    how='inner'
)

print("Final merged shape:", lowkey_features.shape)
print("\nMissing values:")
print(lowkey_features.isnull().sum())

print("\nPreview:")
display(lowkey_features.head())

print("\nDuplicate timestamps:",
      lowkey_features['timestamp_ist'].duplicated().sum())

In [ ]:
lowkey_features['tap_present'] = (
    lowkey_features['tap_count'] > 0
).astype(int)

lowkey_features['scroll_present'] = (
    lowkey_features['scroll_count'] > 0
).astype(int)

lowkey_features['acc_present'] = (
    lowkey_features['acc_sample_count'] > 0
).astype(int)

print("Modality presence:")
print(lowkey_features[
    ['tap_present', 'scroll_present', 'acc_present']
].sum())

print("\nDataset shape:", lowkey_features.shape)

In [ ]:
# STEP 37: Missing-value flags

columns_to_check = [
    'avg_tap_x', 'avg_tap_y',
    'avg_scroll_x', 'avg_scroll_y',
    'avg_scroll_delta_y',
    'x_mean', 'y_mean', 'z_mean',
    'magnitude_mean'
]

for col in columns_to_check:
    lowkey_features[col + '_missing'] = (
        lowkey_features[col].isna().astype(int)
    )

print("Missing flags created!")
print("New shape:", lowkey_features.shape)

print("\nMissing flag totals:")
print(lowkey_features.filter(like='_missing').sum())

In [ ]:
# STEP 38: Export final dataset

lowkey_features = lowkey_features.sort_values(
    'timestamp_ist'
).reset_index(drop=True)

lowkey_features.to_csv(
    'lowkey_ml_features.csv',
    index=False
)

print("Final dataset exported!")
print("Shape:", lowkey_features.shape)
print("File: lowkey_ml_features.csv")

In [ ]:
# STEP 39: Feature Dictionary

feature_dictionary = pd.DataFrame([
    ['tap_count', 'TAP', 'Count of taps per 10-sec window', 'count'],
    ['avg_tap_x', 'TAP', 'Mean tap X coordinate', 'pixels'],
    ['avg_tap_y', 'TAP', 'Mean tap Y coordinate', 'pixels'],
    ['scroll_count', 'SCROLL', 'Count of scroll events', 'count'],
    ['avg_scroll_x', 'SCROLL', 'Mean scroll X position', 'pixels'],
    ['avg_scroll_y', 'SCROLL', 'Mean scroll Y position', 'pixels'],
    ['avg_scroll_delta_y', 'SCROLL', 'Mean vertical scroll movement', 'pixels'],
    ['acc_sample_count', 'ACCELEROMETER', 'Number of sensor samples', 'count'],
    ['x_mean', 'ACCELEROMETER', 'Mean X acceleration', 'm/s²'],
    ['y_mean', 'ACCELEROMETER', 'Mean Y acceleration', 'm/s²'],
    ['z_mean', 'ACCELEROMETER', 'Mean Z acceleration', 'm/s²'],
    ['magnitude_mean', 'ACCELEROMETER', 'Mean acceleration magnitude', 'm/s²'],
    ['magnitude_rms', 'ACCELEROMETER', 'RMS acceleration magnitude', 'm/s²'],
    ['x_range', 'ACCELEROMETER', 'Maximum minus minimum X', 'm/s²'],
    ['y_range', 'ACCELEROMETER', 'Maximum minus minimum Y', 'm/s²'],
    ['z_range', 'ACCELEROMETER', 'Maximum minus minimum Z', 'm/s²'],
    ['tap_present', 'FLAG', 'Whether TAP activity exists', 'binary'],
    ['scroll_present', 'FLAG', 'Whether SCROLL activity exists', 'binary'],
    ['acc_present', 'FLAG', 'Whether accelerometer samples exist', 'binary'],
], columns=['feature_name', 'source', 'description', 'unit'])

feature_dictionary.to_csv('lowkey_feature_dictionary.csv', index=False)

print("Feature dictionary saved!")
display(feature_dictionary)

In [ ]:
# STEP 40: Final validation

print("Dataset shape:", lowkey_features.shape)

print("\nDuplicate rows:",
      lowkey_features.duplicated().sum())

print("\nDuplicate timestamps:",
      lowkey_features['timestamp_ist'].duplicated().sum())

print("\nMissing values:")
print(lowkey_features.isnull().sum())

print("\nInfinite values:",
      np.isinf(
          lowkey_features.select_dtypes(include=np.number)
      ).sum().sum())

print("\nDate range:")
print(lowkey_features['timestamp_ist'].min())
print(lowkey_features['timestamp_ist'].max())

In [ ]:
import zipfile
import os

final_files = [
    'cleaned_tap_scroll.csv',
    'cleaned_accelerometer.csv',
    'lowkey_ml_features.csv',
    'lowkey_feature_dictionary.csv',
    'accelerometer_10sec_features.csv'
]

with zipfile.ZipFile(
    'LOWKEY_Member2_Deliverables.zip', 'w'
) as zipf:

    for file in final_files:
        if os.path.exists(file):
            zipf.write(file)

print("ZIP created successfully!")

for file in final_files:
    print(file, os.path.exists(file))

In [ ]:
import os

for file in os.listdir():
    if 'accelerometer' in file.lower():
        print(file)

In [ ]:
acc_clean.to_csv('cleaned_accelerometer.csv', index=False)

print("Saved:", os.path.exists('cleaned_accelerometer.csv'))

In [ ]:
with zipfile.ZipFile('LOWKEY_Member2_Deliverables.zip', 'w') as zipf:
    for file in [
        'cleaned_tap_scroll.csv',
        'cleaned_accelerometer.csv',
        'lowkey_ml_features.csv',
        'lowkey_feature_dictionary.csv',
        'accelerometer_10sec_features.csv'
    ]:
        zipf.write(file)

print("Final ZIP ready!")

In [ ]:
import os

for file in os.listdir('/content'):
    print(file)